In [ ]:
# COFEU SFT — Türkçe Instruction Tuning (Colab)
# GPU: Colab T4/A100 | ~25-30 dk
#
# Model + vocab Google Drive'da zaten var (cofeuai/checkpoints/),
# yeniden yüklemek gerekmiyor.

In [ ]:
# GPU kontrolü: CPU'da eğitim saatlerce süreceği için baştan uyaralım.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU YOK"
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
import subprocess, sys
r = subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/cofeu/cofeuai.git", "/content/cofeuai"],
                   capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-300:])
if r.returncode != 0:
    raise RuntimeError("repo klonlanamadi (internet/DNS kontrolu)")
print(subprocess.run(["git", "-C", "/content/cofeuai", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive baglandi")

In [ ]:
import os, shutil, urllib.request
from pathlib import Path

REPO = Path('/content/cofeuai')
DRIVE_ROOT = Path('/content/drive/MyDrive/cofeuai')
(REPO / 'checkpoints').mkdir(parents=True, exist_ok=True)
(REPO / 'python').mkdir(parents=True, exist_ok=True)

def on_drive(name):
    """Drive'da dosyayi ara: once standart yol, sonra genis arama."""
    for c in (DRIVE_ROOT / 'checkpoints' / name, DRIVE_ROOT / name):
        if c.exists():
            return c
    hits = list(DRIVE_ROOT.rglob(name)) if DRIVE_ROOT.exists() else []
    return hits[0] if hits else None

# ---- model + vocab: Drive'dan ----
missing = []
for name, dest in [('cofeu.pt', REPO / 'checkpoints' / 'cofeu.pt'),
                   ('vocab.json', REPO / 'checkpoints' / 'vocab.json')]:
    src = on_drive(name)
    if src is None:
        missing.append(name)
        continue
    shutil.copy(src, dest)
    print(f"{name:12s} <- {src}  ({os.path.getsize(dest)/1e6:.1f} MB)")

if missing:
    raise FileNotFoundError(
        "Drive'da bulunamadi: " + ", ".join(missing) +
        "\nBeklenen: /content/drive/MyDrive/cofeuai/checkpoints/cofeu.pt"
        "\ncofeu.pt (353 MB) Drive'da yoksa Colab'a yukleyip su hucreyi tekrar calistirin.")

# ---- sft.py: duzeltilmis surum olmali ----
# Push yapamadigimiz icin GitHub'daki sft.py ESKI (bozuk split'li) olabilir.
# Bu yuzden surumu isaretleyerek dogruluyoruz; eski surum sessizce kullanilmaz.
MARKERS = ["shuffled = list(samples)", "Son model daha kötü"]

src = on_drive('sft.py')
origin = f"Drive ({src})"
if src is None:
    dest = REPO / 'python' / 'sft.py'
    url = "https://raw.githubusercontent.com/cofeu/cofeuai/main/python/sft.py"
    print("sft.py Drive'da yok, GitHub'dan cekiliyor:", url)
    urllib.request.urlretrieve(url, dest)
    origin = f"GitHub raw ({url})"
else:
    dest = REPO / 'python' / 'sft.py'
    shutil.copy(src, dest)

text = dest.read_text(encoding='utf-8')
missing_marks = [m for m in MARKERS if m not in text]
print(f"\nsft.py kaynagi: {origin}  ({len(text)} byte)")
if missing_marks:
    raise RuntimeError(
        "DIUZELTILMIS sft.py BULUNAMADI. Eksik isaretler: " + ", ".join(missing_marks) +
        "\nBu surum val split'ini karistirmiyor ve en iyi checkpoint'i eziyor."
        "\n\nCOZUM: ~/Desktop/CofeuAI/python/sft.py dosyasini Google Drive'a"
        "\nyukleyin: /content/drive/MyDrive/cofeuai/sft.py"
        "\nSonra bu hucreyi tekrar calistirin.")
print("sft.py duzeltmeleri dogrulandi (split + best-checkpoint korumasi)")

In [ ]:
# Veri: once Drive'da donusturulmus turkce_sft.jsonl ara, yoksa HF'den indir.
import os, json, shutil, subprocess
from pathlib import Path

DATA = Path('/content/turkce_sft.jsonl')
DRIVE_ROOT = Path('/content/drive/MyDrive/cofeuai')

if DATA.exists():
    print("veri zaten var")
else:
    hit = None
    for c in (DRIVE_ROOT / 'turkce_sft.jsonl', DRIVE_ROOT / 'data' / 'turkce_sft.jsonl'):
        if c.exists():
            hit = c
            break
    if hit is None and DRIVE_ROOT.exists():
        found = list(DRIVE_ROOT.rglob('turkce_sft.jsonl'))
        hit = found[0] if found else None

    if hit is not None:
        print("Drive'dan aliniyor:", hit)
        shutil.copy(hit, DATA)
    else:
        URL = ("https://huggingface.co/datasets/turkish-nlp-suite/"
               "InstrucTurca/resolve/main/data/train.jsonl")
        print("veri indiriliyor (390 MB, 2-3 dk)...")
        rc = subprocess.run(
            ["curl", "-sL", "--max-time", "900", "-o", "/content/instruk.jsonl", URL]).returncode
        if rc != 0 or not Path('/content/instruk.jsonl').exists():
            raise RuntimeError(
                "veri indirilemedi. Dosyayi Drive'a yukleyin: "
                "/content/drive/MyDrive/cofeuai/turkce_sft.jsonl")
        print("donusturuluyor...")
        out, skip = [], 0
        with open('/content/instruk.jsonl', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    o = json.loads(line)
                except Exception:
                    skip += 1
                    continue
                p = (o.get('Output') or '').strip()
                if not p:
                    skip += 1
                    continue
                out.append({'instruction': (o.get('Input') or '').strip(),
                            'input': '', 'output': p})
        with open(DATA, 'w', encoding='utf-8') as f:
            for r in out:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        print(f"donusturulen: {len(out)} | atlanan: {skip}")

with open(DATA, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
print("ornek sayisi:", n)

In [ ]:
# lr 3e-5: onceki denemelerde 5e-5 bile ezberlemeye yol acti.
# 1 epoch, patience 3: val artinca otomatik durur.
#
# sft.py log DOSYASI yazmiyor (basicConfig yalnizca stream'e).
# Burada ciktiyi hem ekrana (canli) hem Drive'a (kalici) yaziyoruz;
# oturum duserse gecmisi kaybetmeyelim.
import subprocess, sys
from pathlib import Path

OUT_DIR = Path('/content/drive/MyDrive/cofeuai/sft_tr')
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUT_DIR / 'train.log'

cmd = [
    sys.executable, "python/sft.py",
    "--data", "/content/turkce_sft.jsonl",
    "--resume", "/content/cofeuai/checkpoints/cofeu.pt",
    "--out-dir", str(OUT_DIR),
    "--epochs", "1", "--max-samples", "20000",
    "--batch-size", "16", "--lr", "3e-5", "--patience", "3",
    "--eval-interval", "200", "--save-interval", "500",
    "--log-interval", "50", "--device", "cuda",
]
print("calistiriliyor:", " ".join(cmd), "\n")
print(f"log -> {LOG}\n")

with open(LOG, "w", encoding="utf-8", buffering=1) as logf:
    logf.write(f"$ {' '.join(cmd)}\n\n")
    logf.flush()
    p = subprocess.Popen(cmd, cwd="/content/cofeuai",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")      # canli ekrana
        logf.write(line)         # kalici diske
    rc = p.wait()

if rc != 0:
    raise RuntimeError(f"SFT egitimi basarisiz, exit={rc}. Log: {LOG}")
print(f"\nEGITIM TAMAM | log: {LOG}")

In [ ]:
import sys, torch
sys.path.insert(0, '/content/cofeuai/python')
from model import CofeuTransformer, ModelConfig
from tokenizer import BPETokenizer
import sft

CKPT = '/content/drive/MyDrive/cofeuai/sft_tr/cofeu_sft.pt'
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print("iteration:", ck.get('iteration'), "| val:", ck.get('val_loss'))

cfg = ModelConfig(**ck['config'].__dict__)
m = CofeuTransformer(cfg)
m.load_state_dict(ck['model_state'])
m.eval().cuda()
tok = BPETokenizer.load('/content/cofeuai/checkpoints/vocab.json')

for q in ("Merhaba, nasılsın?",
          "Türkiye'nin başkenti neresidir?",
          "Fotosentez nasıl çalışır?",
          "Bana kısa bir motivasyon cümlesi söyle."):
    p, _ = sft.build_text(q, "", "")
    x = torch.tensor([tok.encode(p)], dtype=torch.long).cuda()
    torch.manual_seed(42)
    out = m.generate(x, 120, temperature=0.7, top_k=40, eos_token_id=tok.eos_id)
    print(f"\n>>> {q}\n{tok.decode(out[0, len(x[0]):].tolist())}")

In [ ]:
import subprocess
from pathlib import Path
OUT_DIR = Path('/content/drive/MyDrive/cofeuai/sft_tr')

print(subprocess.run(["bash", "-c", f"tail -25 {OUT_DIR}/train.log"],
                     capture_output=True, text=True).stdout)
print("\n--- dosyalar ---")
print(subprocess.run(["bash", "-c", f"ls -la {OUT_DIR}"],
                     capture_output=True, text=True).stdout)

subprocess.run(["zip", "-j", "/content/cofeu_sft_tr.zip", str(OUT_DIR / 'cofeu_sft.pt')])
print("zip: /content/cofeu_sft_tr.zip  (Colab Files sekmesinden indir)")